# 🏥 Sistema RAG — Correção de Anotações de Enfermagem
> **Atenção:** Execute as células **em ordem**. O cache evita reprocessar o PDF e gastar tokens desnecessariamente.

## Célula 1 — Instalação de dependências
Execute apenas uma vez. Reinicie o kernel depois se necessário.

In [ ]:

%pip install -q numpy PyPDF2 google-generativeai

In [ ]:

!pip install python-dotenv -q

from dotenv import load_dotenv
import os
import google.generativeai as genai
from dotenv import load_dotenv


load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("❌ Erro: A variável 'GOOGLE_API_KEY' não foi encontrada no arquivo .env!")

genai.configure(api_key=api_key)
print("🔒 Chave de API carregada e configurada com segurança via arquivo .env!")

## Célula 2 — Imports e configuração da API

In [ ]:
import numpy as np
import PyPDF2
import json
import hashlib
import time


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

# Alternativa usando a nova geração disponível na sua chave:
model = genai.GenerativeModel('models/gemini-2.5-flash')

print("✅ Configuração carregada.")

## Célula 3 — Funções de chunking e cache

O cache salva os embeddings em disco (`.json`). Na próxima execução, o sistema **detecta o arquivo e pula a vetorização**, economizando chamadas à API.

In [ ]:
# ==========================================
# 📄 EXTRAÇÃO E CHUNKING
# ==========================================

def extrair_e_criar_chunks(caminho_pdf: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """Lê o PDF e divide o texto em blocos com sobreposição. Ignora chunks vazios."""
    if not os.path.exists(caminho_pdf):
        raise FileNotFoundError(f"PDF não encontrado: '{caminho_pdf}'. Verifique o caminho.")

    texto_completo = ""
    with open(caminho_pdf, "rb") as arquivo:
        leitor = PyPDF2.PdfReader(arquivo)
        for i, pagina in enumerate(leitor.pages):
            texto = pagina.extract_text()
            if texto:
                texto_completo += texto + "\n"

    if not texto_completo.strip():
        raise ValueError("Nenhum texto extraído do PDF. O arquivo pode ser escaneado (imagem).")

    chunks = []
    inicio = 0
    while inicio < len(texto_completo):
        fim = inicio + chunk_size
        chunk = texto_completo[inicio:fim].strip()
        if chunk:  # ignora chunks vazios
            chunks.append(chunk)
        inicio += (chunk_size - overlap)

    print(f"📄 PDF processado: {len(chunks)} chunks criados.")
    return chunks


# ==========================================
# 💾 CACHE DE EMBEDDINGS (evita reprocessar)
# ==========================================

def _hash_do_pdf(caminho_pdf: str) -> str:
    """Gera um hash MD5 do conteúdo do PDF para validar o cache."""
    with open(caminho_pdf, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def salvar_cache(caminho_cache: str, chunks: list[str], embeddings: list, hash_pdf: str):
    """Salva chunks e embeddings em disco."""
    dados = {
        "hash_pdf": hash_pdf,
        "chunks": chunks,
        "embeddings": [e for e in embeddings]  # já é lista de floats
    }
    with open(caminho_cache, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False)
    print(f"💾 Cache salvo em '{caminho_cache}'.")


def carregar_cache(caminho_cache: str, hash_pdf: str):
    """Carrega o cache se existir e se o PDF não mudou. Retorna (chunks, embeddings) ou None."""
    if not os.path.exists(caminho_cache):
        return None
    with open(caminho_cache, "r", encoding="utf-8") as f:
        dados = json.load(f)
    if dados.get("hash_pdf") != hash_pdf:
        print("⚠️  PDF mudou desde o último cache. Reprocessando...")
        return None
    print("✅ Cache válido encontrado. Embeddings carregados sem custo de API.")
    return dados["chunks"], dados["embeddings"]


print("✅ Funções de chunking e cache definidas.")

## Célula 4 — Geração de embeddings (com proteção de rate limit)

In [ ]:
# ==========================================
# 🧠 EMBEDDINGS COM RATE LIMIT SEGURO
# ==========================================

def gerar_embeddings_dos_chunks(chunks: list[str], pausa_entre_chunks: float = 0.5) -> list:
    """
    Gera embeddings um por um (correto para a API do Gemini).
    pausa_entre_chunks: segundos de espera entre chamadas para evitar rate limit.
    """
    embeddings = []
    total = len(chunks)
    print(f"🔄 Vetorizando {total} chunks... (isso usa tokens de embedding, não de geração)")

    for i, chunk in enumerate(chunks):
        try:
            response = genai.embed_content(
                model="models/gemini-embedding-001",
                content=chunk,
                task_type="retrieval_document"
            )
            embeddings.append(response["embedding"])

            # Progresso a cada 10 chunks
            if (i + 1) % 10 == 0 or (i + 1) == total:
                print(f"  {i + 1}/{total} chunks vetorizados...")

            time.sleep(pausa_entre_chunks)  # protege contra rate limit

        except Exception as e:
            print(f"  ❌ Erro no chunk {i}: {e}. Pulando...")
            embeddings.append(None)  # mantém alinhamento com chunks

    # Remove chunks com falha de embedding
    pares_validos = [(c, e) for c, e in zip(chunks, embeddings) if e is not None]
    if len(pares_validos) < len(chunks):
        print(f"⚠️  {len(chunks) - len(pares_validos)} chunks descartados por erro.")
    chunks_validos = [p[0] for p in pares_validos]
    embeddings_validos = [p[1] for p in pares_validos]

    print(f"✅ {len(embeddings_validos)} embeddings gerados com sucesso.")
    return chunks_validos, embeddings_validos


print("✅ Função de embedding definida.")

## Célula 5 — Busca por similaridade

In [ ]:
# ==========================================
# 🔍 BUSCA POR SIMILARIDADE DE COSSENO
# ==========================================

def calcular_similaridade(vetor_pergunta: np.ndarray, vetores_documento: np.ndarray) -> np.ndarray:
    """Similaridade de cosseno entre a pergunta e todos os chunks."""
    # Garante tipos numpy corretos
    vetores_documento = np.array(vetores_documento, dtype=np.float32)
    vetor_pergunta = np.array(vetor_pergunta, dtype=np.float32)

    dot_products = np.dot(vetores_documento, vetor_pergunta)
    normas_doc = np.linalg.norm(vetores_documento, axis=1)
    norma_pergunta = np.linalg.norm(vetor_pergunta)

    # Evita divisão por zero
    denominador = normas_doc * norma_pergunta
    denominador = np.where(denominador == 0, 1e-10, denominador)

    return dot_products / denominador


def recuperar_contexto_relevante(pergunta: str, chunks: list[str], vetores_documento: list, top_k: int = 3) -> str:
    """Busca os top_k chunks mais relevantes para a pergunta."""
    response = genai.embed_content(
        model="models/gemini-embedding-001",
        content=pergunta,
        task_type="retrieval_query"
    )
    vetor_pergunta = response["embedding"]

    similaridades = calcular_similaridade(vetor_pergunta, vetores_documento)
    indices_top = np.argsort(similaridades)[::-1][:top_k]

    contexto = "\n---\n".join([chunks[i] for i in indices_top])
    return contexto


print("✅ Funções de busca definidas.")

## Célula 6 — ⚙️ Indexação do PDF

**Execute esta célula apenas quando:**
- For a primeira vez
- Trocar o PDF

Nas demais vezes, o cache é carregado automaticamente **sem custo**.

In [ ]:
# ==========================================
# ⚙️  INDEXAÇÃO COM CACHE AUTOMÁTICO
# ==========================================

CAMINHO_PDF   = "Enviando por email anotacao-de-enfermagem.pdf"  # 
CAMINHO_CACHE = "cache_embeddings.json"        # arquivo gerado automaticamente

# Tenta carregar do cache primeiro
hash_atual = _hash_do_pdf(CAMINHO_PDF)
resultado_cache = carregar_cache(CAMINHO_CACHE, hash_atual)

if resultado_cache:
    # ✅ Cache válido — zero chamadas de embedding
    chunks_cofen, vetores_cofen = resultado_cache
    vetores_cofen_np = np.array(vetores_cofen, dtype=np.float32)
    print(f"📦 {len(chunks_cofen)} chunks carregados do cache. Pronto para uso!")
else:
    # 🔄 Primeira vez ou PDF mudou — processa e salva cache
    chunks_raw = extrair_e_criar_chunks(CAMINHO_PDF)
    chunks_cofen, vetores_cofen = gerar_embeddings_dos_chunks(chunks_raw)
    vetores_cofen_np = np.array(vetores_cofen, dtype=np.float32)
    salvar_cache(CAMINHO_CACHE, chunks_cofen, vetores_cofen, hash_atual)
    print(f"🎉 Indexação concluída! {len(chunks_cofen)} chunks prontos.")

## Célula 7 — Funções de uso do simulador

In [ ]:
# ==========================================
# 🤖 FUNÇÕES DO SIMULADOR
# ==========================================

def sugerir_com_base_no_rag(anotacao_original: str, cargo: str) -> str:
    """
    Revisa uma anotação de enfermagem com base nas normas do COFEN.
    cargo: '1' para Técnico de Enfermagem, '2' para Enfermeiro.
    """
    if cargo not in ("1", "2"):
        raise ValueError("cargo deve ser '1' (Técnico) ou '2' (Enfermeiro).")

    perfil = "TÉCNICO DE ENFERMAGEM" if cargo == "1" else "ENFERMEIRO"
    contexto = recuperar_contexto_relevante(anotacao_original, chunks_cofen, vetores_cofen_np, top_k=3)

    prompt = f"""Normas Técnicas Selecionadas do COFEN:
{contexto}

Perfil do Usuário: {perfil}

Sua tarefa é revisar o registro de saúde abaixo. Aja como revisor de prontuário clínico de enfermagem.
Transforme anotações informais ou com desvios em registros técnicos e legais estritos.

REGRAS:
1. TÉCNICO: Use termos de ANOTAÇÃO (procedimentos realizados,sinais observáveis, queixas). Proibido termos de diagnóstico
   ou exame físico (ex: 'ascite', 'globo vesical', 'murmúrio vesicular').
   Nota: 'distendido' é observação técnica válida de inspeção para técnicos.
2. ENFERMEIRO: Use termos de EVOLUÇÃO (exame físico completo e termos propedêuticos).
3. Se alguma informação faltar (lateralidade, escala de dor etc.), use [especificar].
4. Não inclua introduções, encerramentos informais, assinatura, carimbo ou data.

ESTRUTURA DE RETORNO:
- Sugestão: [texto estruturado]
- Por que: [justificativa técnica em uma linha]

Anotação Original: "{anotacao_original}"""

    response = model.generate_content(prompt)
    return response.text.strip()


def responder_duvida_rag(pergunta: str) -> str:
    """Responde dúvidas sobre legislação de enfermagem com base no PDF indexado."""
    contexto = recuperar_contexto_relevante(pergunta, chunks_cofen, vetores_cofen_np, top_k=4)

    prompt = f"""Contexto Normativo do COFEN:
{contexto}

Você é um Consultor Técnico de Enfermagem especialista em Legislação e Auditoria.
Seja direto e técnico. Responda apenas o que foi perguntado, com base estrita no contexto normativo.

Pergunta: "{pergunta}"""

    response = model.generate_content(prompt)
    return response.text.strip()


print("✅ Funções do simulador prontas.")

## Célula 8 — 🧪 Teste: Correção de anotação

In [ ]:
cargo = input("Você é:\n  1 - Técnico de Enfermagem\n  2 - Enfermeiro\nDigite 1 ou 2: ").strip()

while cargo not in ("1", "2"):
    cargo = input("❌ Opção inválida. Digite 1 ou 2: ").strip()

anotacao = input("\nDigite a anotação para corrigir:\n> ").strip()

while not anotacao:
    anotacao = input("❌ Anotação vazia. Tente novamente:\n> ").strip()

print("\n⏳ Processando...\n")
resultado = sugerir_com_base_no_rag(anotacao, cargo)
print("=" * 60)
print(resultado)
print("=" * 60)

## Célula 9 — 🧪 Teste: Dúvida normativa

In [ ]:
# =====================================================================
# Célula 9 — 🧪 Teste: Dúvida normativa dinâmica
# =====================================================================

duvida = input("❓ Digite sua dúvida sobre as normativas do COFEN:\n> ").strip()

while not duvida:
    duvida = input("❌ A dúvida não pode estar vazia. Tente novamente:\n> ").strip()

print("\n⏳ Consultando a base normativa RAG...\n")

# Executa a busca semântica e gera a resposta fundamentada
resposta = responder_duvida_rag(duvida)

print("=" * 70)
print(resposta)
print("=" * 70)